# Titanic Survival Prediction — Exploratory Analysis to Machine Learning Pipeline

## 📌 Executive Overview
This notebook demonstrates an end-to-end Machine Learning workflow for predicting passenger survival on the Titanic dataset (Kaggle Competition).

### Workflow Stages:
1. **Data Loading & Sanity Checks**
2. **Exploratory Data Analysis (EDA)**
3. **Missing Value Imputation without Data Leakage**
4. **Domain-Specific Feature Engineering (Title, Family Structure, Cabin Decks)**
5. **Model Training & 5-Fold Stratified Cross-Validation Benchmark**
6. **Hyperparameter Tuning & Soft Voting Ensembling**
7. **Submission File Generation (`submission.csv`)**

In [ ]:
import sys
sys.path.append("..")
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_prep import load_data, impute_missing
from src.features import prepare_modeling_data
from src.evaluate import compute_metrics, plot_survival_by_demographics, plot_correlation_matrix
from src.train import evaluate_models_cv, train_and_tune_best_model

%matplotlib inline

## 1. Data Loading

In [ ]:
train_df, test_df = load_data(data_dir="../data/raw")
print(f"Train Data Shape: {train_df.shape}")
print(f"Test Data Shape:  {test_df.shape}")
train_df.head()

## 2. Missing Value Analysis & EDA

In [ ]:
print("Train Missing Values:")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])
print("\nTest Missing Values:")
print(test_df.isnull().sum()[test_df.isnull().sum() > 0])

## 3. Data Imputation & Feature Engineering

In [ ]:
train_clean, test_clean = impute_missing(train_df, test_df)
X_train, y_train, X_test, t_ids, te_ids, train_eng, test_eng = prepare_modeling_data(train_clean, test_clean)
print(f"Engineered X_train shape: {X_train.shape}")
print(f"Engineered Features: {list(X_train.columns)}")

## 4. Stratified 5-Fold Cross-Validation Benchmark

In [ ]:
cv_results = evaluate_models_cv(X_train, y_train)
cv_results

## 5. Hyperparameter Tuning & Soft Voting Ensembling

In [ ]:
best_model, model_name, best_score = train_and_tune_best_model(X_train, y_train)
print(f"Final Selected Best Model: {model_name} with CV Accuracy: {best_score:.4f}")

## 6. Generate Kaggle Submission File

In [ ]:
preds = best_model.predict(X_test)
sub = pd.DataFrame({"PassengerId": te_ids, "Survived": preds})
sub.to_csv("../submission.csv", index=False)
print(f"Generated submission with {len(sub)} rows.")
sub.head(10)